In [ ]:
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
import pymc as pm
import arviz as az
import pandas as pd
from collections import defaultdict, Counter
from dataclasses import dataclass, field
from typing import Optional
import random
import itertools
import warnings

# Notebook config
warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
%matplotlib inline

# Board Game Simulation

## Volcano Rush

### Abstract

*(To be filled at the end of the project)*

- Summary of simulation approach and scope
- Key findings: win rates, character balance, resource dynamics
- Main conclusions on game design

### Introduction

- Board game balance is hard to test through playtesting alone — too many configurations (player count 4–8, character combos, strategy variance)
- A well-balanced game needs a group win rate that feels achievable but not trivial, and individual scores not dominated by one character
- Research questions:
  - How does win rate change across player counts (4 vs 6 vs 8)?
  - Which character roles give the strongest individual scoring advantage?
  - How tight is the resource economy on average?
  - How often does the volcano erupt before escape?
- Monte Carlo simulation as a scalable alternative to physical playtesting
- Semi-cooperative tension (shared win, individual ranking) as an interesting design space

### Related Work

- Monte Carlo methods applied to board games (Monopoly property value, Pandemic infection spread modeling)
- Agent-Based Modeling (ABM) for emergent group behavior — relevant because characters have heterogeneous abilities
- Cooperative game theory: when do individual incentives conflict with group survival?
- Simulation studies of deck-based stochastic games
- Prior work on game balance quantification (win-rate parity, character dominance metrics)
- *(To be expanded with actual citations)*

### Data

- No external dataset — all data is synthetically generated by the simulation
- Game parameter sources:
  - Mission pool (8 standard + 5 boat missions) from `docs/game_rules.md`
  - Complication deck (11 cards) — draw probabilities derived from deck composition
  - Volcano deck (11 cards + Eruption at bottom)
  - Character roles (6) with defined abilities
  - Player count configurations: 4, 5, 6, 7, 8
- Simulation assumptions documented in `docs/simulation_assumptions.md` (exhaustion timing, Craftsman repair cycle, Sailor severity scoring, etc.)
- Each simulation run = one full game; N runs per configuration = our dataset
- Derived output variables: win/loss, rounds elapsed, individual scores, boat parts completed, volcano cards remaining

### Methods

- Monte Carlo simulation: run thousands of games under fixed strategy assumptions
- Agent strategy modeled as rule-based heuristics — e.g., always participate if resources allow
- Sensitivity analysis: vary strategy aggressiveness, player count, character assignment
- Bayesian inference (PyMC) to estimate win-rate distributions with uncertainty

#### Define Game State & Parameters

- Dataclasses / enums for: `Player`, `Character`, `Mission`, `ComplicationCard`, `VolcanoCard`, `GameState`
- Parameters: player count, character assignments, deck compositions, resource deck size
- Strategy parameter: participation threshold (minimum resources a player needs to decide to join a mission)

In [ ]:
# Game state & data structure definitions


#### Implement Simulation Engine

- Round loop: 6 steps (mission selection → participation → complication draw → resolution → exhaustion → mission refresh)
- Mission resolution: count participants, check tool availability, apply character abilities, check success
- Handle special complications (Night Anxiety requires a non-participant helper, Camp Panic limits resources per player, etc.)
- Handle volcano effects (Rain and Mud cost increases, Heat Wave extended exhaustion, Lava Flow removes missions, etc.)
- Terminal conditions: Eruption card drawn (loss) or all required boat missions complete (win)

In [ ]:
# Simulation engine


#### Configure Simulation Scenarios

- Scenarios by player count: 4, 5, 6, 7, 8
- Character composition: random assignment vs. fixed "strong" team vs. fixed "weak" team
- Strategy variants: aggressive (always participate) vs. conservative (gather-heavy)
- Number of runs per scenario: N = 1,000–10,000 (tune based on variance)

In [ ]:
# Scenario configuration & run loop


#### Exploratory Simulation Runs

- Single-game trace to verify logic (print round-by-round state)
- Check distribution of game lengths (rounds to win or lose)
- Sanity checks: resource deck exhaustion rate, average volcano cards drawn per game

In [ ]:
# Exploratory runs + sanity checks


#### Visualize Outcomes

- Win rate by player count (bar chart with confidence intervals)
- Distribution of game length (histogram)
- Character score distributions (violin or box plot per character)
- Volcano deck depth at game end (how close to eruption)
- Resource economy: average cards in hand per round
- Mission completion heatmap (which missions complete most often)

In [ ]:
# Plots


#### Statistical Analysis

- Bayesian estimation of win rate per scenario using PyMC Beta-Binomial model
- Posterior comparison: does player count 6 win significantly more often than 4?
- Character dominance: Bayesian comparison of individual scores per character
- ArviZ for posterior visualization and credible intervals

In [ ]:
import pymc as pm
import arviz as az

# PyMC models + ArviZ plots

### Results

- Win rates across player counts — is it easier with more players?
- Which characters emerge as score leaders? Is there a dominant strategy?
- Resource bottlenecks: which resource type runs out most often?
- How often does the volcano erupt vs. the group escaping?
- Boat mission difficulty ranking (which cause the most failures?)
- Complication card impact — which ones most often flip a win to a loss?

### Discussion / Further Work

- Is the game balanced as designed, or does it favor a specific player count?
- Suggestions for rule tweaks to improve balance (adjust required boat parts, resource deck size)
- Limitations: heuristic strategies don't capture human creativity or negotiation
- Next steps:
  - Replace rule-based agents with learned strategies (RL or MCTS)
  - Explore asymmetric information (players don't see each other's hands)
  - Sensitivity to rule variants (e.g., exhaustion only on success)
  - Compare simulated win rates against real playtest results

### References / Bibliography

- *(To be filled with actual citations)*
- PyMC documentation
- ArviZ documentation
- Relevant paper on Monte Carlo board game simulation
- ABM reference (e.g., Mesa framework)

### Appendix

- **Appendix A:** Full mission table (from `docs/game_rules.md`)
- **Appendix B:** Complication card descriptions
- **Appendix C:** Volcano card descriptions
- **Appendix D:** Character ability reference table
- **Appendix E:** Simulation assumptions (cross-ref `docs/simulation_assumptions.md`)
- **Appendix F:** Raw simulation output tables